In [ ]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Load data from Kaggle (or local path)
url = "https://raw.githubusercontent.com/ashishpatel26/Training-data/master/Housing%20Price%20Prediction/train.csv"
df = pd.read_csv(url)

# Step 1: Preprocessing
# Drop ID column (not useful for prediction)
df.drop('Id', axis=1, inplace=True)

# Fill missing values (numeric: median; categorical: mode)
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include=['object']).columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Encode categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns
le_dict = {}  # Store encoders per column for future use
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le  # Save encoder for later inverse transform if needed

# Define X (features) and y (target)
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']  # Continuous target → REGRESSION

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize Random Forest Regressor with ALL key parameters explained
rf_reg = RandomForestRegressor(
    n_estimators=200,            # 👉 More trees → better stability (default=100)
    criterion='squared_error',   # 👉 For regression: 'squared_error' (MSE), 'absolute_error', 'poisson'
    max_depth=15,                # 👉 Max tree depth → prevent overfitting
    min_samples_split=10,        # 👉 Min samples to split node → avoid tiny splits
    min_samples_leaf=5,          # 👉 Min samples in leaf → smooth predictions
    max_features='auto',         # 👉 For regression: 'auto' = all features (default); 'sqrt' also common
    bootstrap=True,              # 👉 Use bootstrap sampling → enables OOB error
    oob_score=True,              # 👉 Compute OOB score during fit → internal validation
    random_state=42,             # 👉 Reproducible results
    n_jobs=-1,                   # 👉 Use all CPU cores → faster training
    verbose=1                    # 👉 Show progress (optional)
)

# Fit the model
rf_reg.fit(X_train, y_train)

# Predict
y_pred = rf_reg.predict(X_test)

# Evaluate Regression Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

# Optional: Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_reg.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Optional: OOB Score (if bootstrap=True)
print(f"\nOOB R² Score: {rf_reg.oob_score_:.4f}")